# Check / resume OneDrive → Google Drive

Upload the same `rclone.conf` used for the migration, then run the single code cell below.

It checks `onedrive-src:` against `gdrive-dst:OneDrive Migration` first. If everything is present with the same size, it stops without copying. If anything is missing or size-mismatched, it resumes with `rclone copy` and verifies again.


In [ ]:
import shutil
import subprocess
from pathlib import Path
from google.colab import files

SOURCE = "onedrive-src:"
DESTINATION = "gdrive-dst:OneDrive Migration"
CONFIG = "/content/rclone.conf"

def run_rclone(*args, check=True):
    cmd = ["rclone", *args, "--config", CONFIG]
    return subprocess.run(cmd, check=check)

def line_count(path):
    p = Path(path)
    if not p.exists():
        return 0
    with p.open("r", encoding="utf-8", errors="replace") as f:
        return sum(1 for _ in f)

def verify(tag):
    missing = f"/content/{tag}-missing-on-dst.txt"
    differ = f"/content/{tag}-different.txt"
    errors = f"/content/{tag}-errors.txt"
    for report in (missing, differ, errors):
        Path(report).unlink(missing_ok=True)

    result = run_rclone(
        "check", SOURCE, DESTINATION,
        "--one-way",
        "--size-only",
        "--missing-on-dst", missing,
        "--differ", differ,
        "--error", errors,
        "--checkers", "8",
        check=False,
    )
    counts = {
        "missing": line_count(missing),
        "different_size": line_count(differ),
        "errors": line_count(errors),
    }
    print(f"Check result: missing={counts['missing']}, different_size={counts['different_size']}, errors={counts['errors']}")
    return result.returncode, (missing, differ, errors)

def preview_reports(reports, limit=30):
    for report in reports:
        p = Path(report)
        if p.exists() and p.stat().st_size:
            print(f"\n--- {p.name} (first {limit}) ---")
            with p.open("r", encoding="utf-8", errors="replace") as f:
                for i, line in enumerate(f):
                    if i >= limit:
                        print("...")
                        break
                    print(line.rstrip())

if shutil.which("rclone") is None:
    print("Installing rclone...")
    subprocess.run(["bash", "-lc", "curl -fsSL https://rclone.org/install.sh | sudo bash"], check=True)

subprocess.run(["rclone", "version"], check=True)

print("Upload rclone.conf")
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one rclone.conf file.")
name, data = next(iter(uploaded.items()))
if Path(name).name.lower() != "rclone.conf":
    raise RuntimeError("The uploaded file must be named rclone.conf.")
Path(CONFIG).write_bytes(data)
Path(CONFIG).chmod(0o600)

configured = subprocess.run(
    ["rclone", "listremotes", "--config", CONFIG],
    check=True, capture_output=True, text=True,
).stdout.splitlines()
for required in ("onedrive-src:", "gdrive-dst:"):
    if required not in configured:
        raise RuntimeError(f"Missing required remote: {required}")

print("Validating remotes...")
run_rclone("lsf", SOURCE, "--max-depth", "1")
run_rclone("lsf", "gdrive-dst:", "--max-depth", "1")

print("\nChecking current migration...")
before_code, before_reports = verify("before")

if before_code == 0:
    print("\n✅ COMPLETE: OneDrive Migration already contains every OneDrive file with the same size. No copy needed.")
else:
    print("\n⚠️ Migration is incomplete or has mismatches. Resuming copy...")
    preview_reports(before_reports, limit=20)
    run_rclone(
        "copy", SOURCE, DESTINATION,
        "--size-only",
        "--stats", "10s",
        "--stats-one-line",
        "--stats-one-line-date",
        "--transfers", "4",
        "--checkers", "8",
        "--create-empty-src-dirs",
    )

    print("\nCopy finished. Verifying again...")
    after_code, after_reports = verify("after")
    if after_code == 0:
        print("\n✅ COMPLETE: resume finished and verification passed.")
    else:
        preview_reports(after_reports, limit=50)
        raise RuntimeError("Final verification FAILED. See the report previews above and files under /content/after-*.txt.")
